In [1]:
!pip install librosa tensorflow numpy matplotlib

In [2]:
import librosa
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Training Data
Can be found here: https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset

The data is not stored in the github for file size reasons. The dataset needs to be downloaded for code to function properly.

In [3]:
# Only using 2-sec folder of the data since we are only examining short form audio we 
FAKE_TRAINING_DATA = "archive/for-2sec/for-2seconds/training/fake"
REAL_TRAINING_DATA = "archive/for-2sec/for-2seconds/training/real"

FAKE_TESTING_DATA = "archive/for-2sec/for-2seconds/testing/fake"
REAL_TESTING_DATA = "archive/for-2sec/for-2seconds/testing/real"

FAKE_VALIDATION_DATA = "archive/for-2sec/for-2seconds/validation/fake/"
REAL_VALIDATION_DATA = "archive/for-2sec/for-2seconds/validation/real/"

In [4]:
def process_for_2sec_file(file_path):
    # 'duration=1.0' tells librosa to only load the first second  of the 2-second file in the for-2sec folder.
    y, sr = librosa.load(file_path, sr=22050, duration=1.0)
    
    # in case a file is slightly under 1 second, fix the length
    y = librosa.util.fix_length(y, size=sr)
    
    # Create melspectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    
    
    log_S = librosa.power_to_db(S)
    
    delta_S = librosa.feature.delta(log_S)
    
#     return log_S
    return np.stack([log_S, delta_S], axis=-1)


def iterate_folder(data_folder):
    data_list = []

    #files = os.listdir(data_folder)[:1000]
    files = os.listdir(data_folder)
   
    for filename in tqdm(files, desc=f"Processing {os.path.basename(data_folder)}", unit="file"):
        file_path = os.path.join(data_folder, filename)

        # Only use .wav files
        if file_path.endswith(".wav"):
            result = process_for_2sec_file(file_path)
            data_list.append(result)
    return data_list

# Get list of real and fake training data
fake_train_data, real_train_data = iterate_folder(FAKE_TRAINING_DATA), iterate_folder(REAL_TRAINING_DATA)

# Get list of real and fake test data
fake_test_data, real_test_data = iterate_folder(FAKE_TESTING_DATA), iterate_folder(REAL_TESTING_DATA)

# Get list of real and fake validation data
fake_valid_data, real_valid_data = iterate_folder(FAKE_VALIDATION_DATA), iterate_folder(REAL_VALIDATION_DATA)

Processing fake:   0%|          | 0/6978 [00:00<?, ?file/s]C:\ProgramData\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,
Processing : 100%|██████████| 1413/1413 [00:11<00:00, 122.74file/s]


# Create Answer Key
We need to assign an answer key to the training data. In this case, 0's will indicate real audio files, 1's will indicate fake audio files

In [5]:
def convert_to_numpy(real, fake):
    # Convert lists to numpy arrays
    X_real = np.array(real)
    X_fake = np.array(fake)

    # Create labels (0s for real, 1s for fake)
    y_real = np.zeros(len(X_real))
    y_fake = np.ones(len(X_fake))

    # Stack them together
    X = np.concatenate((X_real, X_fake), axis=0)
    y = np.concatenate((y_real, y_fake), axis=0)

    return X, y

X_train, y_train = convert_to_numpy(real_train_data, fake_train_data)
X_test, y_test = convert_to_numpy(real_test_data, fake_test_data)
X_val, y_val = convert_to_numpy(real_valid_data, fake_valid_data)


# Making the Spectrogram 4D
CNNs need to have a 4D format, therefore we need to add an extra parameter to each spectrogram

In this case the four dimensions are:
1. Number of 1-Second iterations
2. Frequency Resolution (Mel Bins)
3. Time Resolution (1 second divded by hop length)
4. Color Channel Intensity ()

In [6]:
# def reshape(X):
#     # Reshape from (2000, 128, 44) to (2000, 128, 44, 1)
#     return X.reshape(X.shape[0], X.shape[1], X.shape[2], 1)

# X_train, X_test, X_val = reshape(X_train), reshape(X_test), reshape(X_val)

In [7]:
# # Assuming X is your final combined and reshaped array
# print("--- Data Specification Summary ---")
# print(f"Total Samples (1st Dim):  {X.shape[0]}")
# print(f"Frequency Bins (2nd Dim): {X.shape[1]}")
# print(f"Time Frames (3rd Dim):    {X.shape[2]}")
# print(f"Channels (4th Dim):       {X.shape[3]}")
# print("----------------------------------")
# print(f"Final 4D Shape: {X.shape}")

# Normalize the Data
The numbers are too large to be efficiently computed by a CNN, we need to normalize the data now

In [8]:
# 1. Calculate the 'Master' Min and Max from Training ONLY
# train_min = X_train.min()
# train_max = X_train.max()
 
# # 2. Define a function that takes these master values
# def apply_master_normalization(data, master_min, master_max):
#     return (data - master_min) / (master_max - master_min)

# # 3. Apply the SAME scaling to all three sets
# X_train = apply_master_normalization(X_train, train_min, train_max)
# X_val   = apply_master_normalization(X_val, train_min, train_max)
# X_test  = apply_master_normalization(X_test, train_min, train_max)

# # outlier removal?



# def normalize_per_sample(X):
#     # Create an empty array of the same shape
#     X_norm = np.zeros_like(X)
#     for i in range(len(X)):
#         sample = X[i]
#         s_min, s_max = sample.min(), sample.max()
#         if s_max - s_min > 0:
#             X_norm[i] = (sample - s_min) / (s_max - s_min)
#     return X_norm

# X_train = normalize_per_sample(X_train)
# X_val = normalize_per_sample(X_val)
# X_test = normalize_per_sample(X_test)


In [9]:
# from sklearn.utils import shuffle

# # Shuffle Training Data
# X_train, y_train = shuffle(X_train, y_train, random_state=42)

# # Shuffle Validation Data
# X_val, y_val = shuffle(X_val, y_val, random_state=42)

# # Shuffle Testing Data
# X_test, y_test = shuffle(X_test, y_test, random_state=42)

# Save the data so other notebooks can access it

In [10]:
# Save X and y into a single compressed file
np.savez_compressed('processed_data.npz', 
                    x_train=X_train,
                    y_train=y_train,
                    x_val=X_val,
                    y_val=y_val,
                    x_test=X_test,
                    y_test=y_test)
print("Successfully saved to 'processed_data.npz'")



Successfully saved to 'processed_data.npz'
